# Flows · **11 — Findings: what the flows support, and what they don't**

The same shape as `08_findings`: each claim the flows piece might make, beside
the rebuilt figure, its basis, a verdict from a fixed vocabulary, and the files
behind it. **Only "holds" goes forward as a claim.**

| verdict | meaning |
|---|---|
| **holds** | rebuilt from the files (or, for a claim about a document, read in it) and true as worded |
| **true but misleading** | the number is right; the inference drawn from it isn't |
| **overstated** | the data supports a weaker version |
| **does not hold** | contradicted by the files or the definitions |
| **sourced, not rebuilt** | NHS England's own figure, true to its publication, but not something I can rebuild |

Every number comes from notebook 10's Gold (which passed its 16-check flows
gate) or from the evidence saved in `data/bronze/nhse_evidence_flows_2026-11/`.
Each verdict comes from an explicit rule beside it. Headline basis (with NHS
England's estimates for missing trusts) unless a row says otherwise. Flows are
quoted to the nearest thousand and shares to one decimal, per notebook 10's
rounding rule.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR, REPORT_MONTH, month_year  # noqa: E402

OUT = GOLD_DIR / "11_flows_findings"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 90)
EV = "data/bronze/nhse_evidence_flows_2026-11/"
F_EXPL = EV + "20260414-Unreported-Removals-Summary.docx"
F_GUIDE = EV + "NHSE_RTT_statistics_user_guidance_page.html"
F_OSR = EV + "OSR_Siobhan_Tuohy-Smith_to_Chris_Roebuck_RTT_CR_implementing_recommendations_2026-11-06.pdf"
F_NHSE = EV + "NHSE_Chris_Roebuck_to_Siobhan_Tuohy-Smith_RTT_CR_addressing_recommendations_2026-10-21.pdf"
F_OV = f"data/bronze/nhse_rtt_published_{REPORT_MONTH}/RTT-Overview-Timeseries-*.xlsx"
G_FLOWS, G_PROV = "10_flows/flows_national.csv", "10_flows/provider_unreported.csv"

def k(x):    # nearest thousand, as quoted in prose
    return f"{round(x, -3):,.0f}"

## Step 0 — Both gates must have passed

In [2]:
main = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
flows_gate = pd.read_csv(GOLD_DIR / "10_flows" / "flows_gate.csv")
assert main["pass"].all() and flows_gate["pass"].all(), "run 04 and 10 first: a gate is failing"
f = pd.read_csv(GOLD_DIR / G_FLOWS)
h = f[f.basis == "headline"].set_index("period")
r = f[f.basis == "reported"].set_index("period")
prov = pd.read_csv(GOLD_DIR / G_PROV)
assert h.index[-1] == REPORT_MONTH
print(f"main gate {len(main)}/{len(main)}, flows gate {len(flows_gate)}/{len(flows_gate)}; window {h.index[0]} to {h.index[-1]}")

main gate 52/52, flows gate 16/16; window 2025-04 to 2026-09


## Step 1 — The quantities the claims rest on

In [3]:
u = h.unreported_published                        # derived unreported removals, headline
c = h.completed                                   # reported completed pathways, headline
share = 100 * u / (u + c)                         # unreported as % of all exits (NHS England's 'total removals')
MAR, SEP = "2026-03", REPORT_MONTH
med_u, med_s = u.median(), share.median()
rank = u.rank(ascending=False).astype(int)
pre = [p for p in h.index if p < MAR]
post = [p for p in h.index if p > MAR]
pooled = lambda ps: 100 * u[ps].sum() / (u[ps].sum() + c[ps].sum())
summary = pd.DataFrame({"unreported": u, "completed": c, "share_pct": share, "rank_by_level": rank,
                        "vs_median": u / med_u, "net_change": h.net_change})
summary.to_csv(OUT / "monthly_unreported.csv")
summary.round(2)

,unreported,completed,share_pct,rank_by_level,vs_median,net_change
period,,,,,,
2025-04,286883.00,1450094.00,16.52,5,1.10,-29533.0
2025-05,289637.00,1500695.00,16.18,4,1.11,-33115.0
2025-06,257791.00,1558632.00,14.19,11,0.99,11886.0
2025-07,271149.81,1648944.52,14.12,7,1.04,22443.0
2025-08,209442.10,1404088.24,12.98,18,0.80,13333.0
2025-09,243721.90,1617382.76,13.10,14,0.93,-15463.0
2025-10,245036.81,1677868.52,12.74,12,0.94,7296.0
2025-11,260299.31,1536543.11,14.49,10,1.00,-86517.0
2025-12,238198.00,1426850.00,14.31,16,0.91,-18048.0


## Step 2 — The rows

In [4]:
rows = []
def row(section, claim, rebuilt, basis, verdict, rule, gold, primary):
    rows.append({"section": section, "claim_as_stated": claim, "rebuilt": rebuilt, "basis": basis,
                 "verdict": verdict, "rule": rule, "gold_file": gold, "primary_file": primary})

# --- monthly level and share -------------------------------------------------
row("Monthly flows",
    f"Each month from {month_year(h.index[0])} to {month_year(SEP)}, NHS England's figures show about "
    f"{k(u.min())}-{k(u.max())} more pathways leaving the list than were reported as completed",
    f"{u.min():,.0f} ({u.idxmin()}) to {u.max():,.0f} ({u.idxmax()}); median {med_u:,.0f}",
    "headline (derived balancing item)", "holds",
    "rebuilt by the identity in 10 (F1 closes every month)", G_FLOWS, F_OV)
row("Monthly flows",
    "Those unreported removals were about one exit in seven",
    f"median {med_s:.1f}% of all exits; range {share.min():.1f}% ({share.idxmin()}) to {share.max():.1f}% ({share.idxmax()}); "
    f"pooled over the window {pooled(list(h.index)):.1f}%",
    "headline", "holds" if 13 <= pooled(list(h.index)) <= 15.5 else "does not hold",
    "holds if the pooled share is between 13% and 15.5% (one in seven is 14.3%)", G_FLOWS, F_OV)
row("Monthly flows",
    "That is in line with NHS England's own long-run figure (14% of removals since 2021)",
    f"window pooled {pooled(list(h.index)):.1f}% vs NHS England's stated 14%",
    "headline vs NHS England explainer", "holds" if abs(pooled(list(h.index)) - 14) <= 1 else "does not hold",
    "holds if within 1 point of the explainer's 14%", G_FLOWS, F_EXPL)

# --- March 2026 -----------------------------------------------------------------
cont = prov[prov.kind == "continuing"].groupby("period").unreported.sum()
cont_med = cont.median()
row("March 2026",
    "March 2026 had the most unreported removals of any month in the window",
    f"{u[MAR]:,.0f}, rank {rank[MAR]} of {len(u)}; {u[MAR] / med_u:.2f}x the window median ({med_u:,.0f})",
    "headline", "holds" if rank[MAR] == 1 else "does not hold", "holds if March ranks 1st by level", G_FLOWS, F_OV)
row("March 2026",
    "March's spike is not a submission artefact: it shows among trusts that reported in both February and March",
    f"continuing trusts {cont[MAR]:,.0f}, rank {int(cont.rank(ascending=False)[MAR])} of {len(cont)}; "
    f"{cont[MAR] / cont_med:.2f}x their median ({cont_med:,.0f})",
    "reported, continuing trusts only", "holds" if cont.rank(ascending=False)[MAR] == 1 else "does not hold",
    "holds if March also ranks 1st among continuing trusts", G_PROV, "full CSV extracts (Bronze)")
row("March 2026",
    "In March, unreported removals were also an unusually large share of exits",
    f"{share[MAR]:.1f}% vs median {med_s:.1f}%; rank {int(share.rank(ascending=False)[MAR])} of {len(share)} by share",
    "headline", "holds" if share.rank(ascending=False)[MAR] <= 2 else "does not hold",
    "holds if March ranks in the top two by share", G_FLOWS, F_OV)
excess = u[MAR] - u.drop(MAR).median()
fall = -h.net_change[MAR]
row("March 2026",
    "Arithmetic, not cause: had March's unreported removals been at the other months' median, most of the list's fall that month would not have happened",
    f"March excess over the other months' median: {excess:,.0f}; the list fell {fall:,.0f}; excess = {100 * excess / fall:.0f}% of the fall",
    "headline", "holds" if excess / fall > 0.5 else "does not hold",
    "holds if the excess exceeds half the fall; it is a counterfactual sum, not a causal claim", G_FLOWS, F_OV)
row("March 2026",
    "March was a validation push",
    "not observable: NHS England's explainer says unreported removals are not the same as validation, and validated removals can also be reported as completions",
    "definitions", "does not hold",
    "no published data separates validation from other unreported removals", "-", F_EXPL)

# --- September 2026 -----------------------------------------------------------------
row("September 2026",
    f"In {month_year(SEP)}, unreported removals were above the window median",
    f"{u[SEP]:,.0f} ({u[SEP] / med_u:.2f}x the median), {share[SEP]:.1f}% of exits; rank {rank[SEP]} of {len(u)}",
    "headline (includes estimated trusts)", "holds" if u[SEP] > med_u else "does not hold",
    "holds if above the median", G_FLOWS, F_OV)
row("September 2026",
    f"The reported-providers figure for {month_year(SEP)} ({r.unreported_published[SEP]:,.0f}) overstates unreported removals",
    f"reported {r.unreported_published[SEP]:,.0f} vs headline {u[SEP]:,.0f}; Nottingham (RX1) stopped submitting, booking its "
    f"{prov[(prov.period == SEP) & (prov.entity == 'RX1')].unreported.sum():,.0f}-pathway list as a 'removal'",
    "reported vs headline", "holds" if r.unreported_published[SEP] > u[SEP] else "does not hold",
    "holds if reported exceeds headline in the month a large trust stopped submitting", G_PROV, F_OV)

# --- before and after March ---------------------------------------------------------
pre_s, post_s = pooled(pre), pooled(post)
inside = share[pre].min() <= post_s <= share[pre].max()
row("After the deadline",
    "After March 2026 a larger share of exits went unreported",
    f"pooled: before March {pre_s:.1f}% ({len(pre)} months), after {post_s:.1f}% ({len(post)} months); "
    f"before-March monthly range {share[pre].min():.1f}-{share[pre].max():.1f}%",
    "headline", "does not hold" if inside else "holds",
    "holds only if the after-March pooled share lies outside the before-March monthly range", G_FLOWS, F_OV)
row("After the deadline",
    "The share of exits that went unreported was much the same before and after the deadline",
    f"{pre_s:.1f}% before, {post_s:.1f}% after (difference {post_s - pre_s:+.1f} points)",
    "headline", "holds" if inside else "does not hold",
    "holds if the after-March pooled share lies inside the before-March monthly range", G_FLOWS, F_OV)

# --- provider concentration (continuing trusts only) ------------------------------
pc = prov[prov.kind == "continuing"]
# Grouped by code only: a few trusts have no name in the incomplete table, and
# grouping on a missing name would silently drop their rows.
tot = pc.groupby("entity").unreported.sum().sort_values(ascending=False)
NAMES = pc.dropna(subset=["name"]).drop_duplicates("entity", keep="last").set_index("entity").name
net_total, gross_pos = tot.sum(), tot[tot > 0].sum()
top10 = tot.head(10)
row("Concentration",
    "Unreported removals are spread across many trusts, not driven by a handful",
    f"top 10 of {len(tot)} trusts: {100 * top10.sum() / gross_pos:.0f}% of all positive trust totals "
    f"({100 * top10.sum() / net_total:.0f}% of the net national figure); {int((tot < 0).sum())} trusts net negative over the window",
    "reported, continuing trusts, May 2025 - " + month_year(SEP), "holds" if top10.sum() / gross_pos < 0.4 else "does not hold",
    "holds if the top 10 hold under 40% of the positive totals", G_PROV, "full CSV extracts (Bronze)")
pm = pc[pc.period == MAR].set_index("entity").unreported
base = pc[pc.period != MAR].groupby("entity").unreported.median()
ex = (pm - base.reindex(pm.index).fillna(0)).sort_values(ascending=False)
ex_pos = ex[ex > 0]
overlap = len(set(ex_pos.head(10).index) & set(top10.index))
row("Concentration",
    "March's spike came from the same trusts that drive unreported removals generally",
    f"{overlap} of the 10 trusts with the largest March excess (over their own median) are among the 10 largest contributors overall; "
    f"top 10 March excesses = {100 * ex_pos.head(10).sum() / ex_pos.sum():.0f}% of all positive March excess",
    "reported, continuing trusts", "holds" if overlap >= 5 else "does not hold",
    "holds if at least 5 of March's top 10 are in the overall top 10", G_PROV, "full CSV extracts (Bronze)")
row("Concentration",
    "March's spike was broad, not a few trusts",
    f"{int((ex > 0).sum())} of {len(ex)} continuing trusts were above their own median in March; "
    f"the top 10 supplied {100 * ex_pos.head(10).sum() / ex_pos.sum():.0f}% of the positive excess",
    "reported, continuing trusts", "holds" if ex_pos.head(10).sum() / ex_pos.sum() < 0.5 else "does not hold",
    "holds if the top 10 supplied under half of the positive excess", G_PROV, "full CSV extracts (Bronze)")

# --- what the definitions allow ------------------------------------------------------
row("Definitions",
    "A reader can point at a month and say how much of the fall was treatment",
    "no: reported completions include clock stops that are not treatment, and unreported removals are a balancing item mixing late-recorded completions, nullifications, duplicates and data errors",
    "definitions", "does not hold",
    "the chart can show reported against unreported, not treated against bookkeeping", "-", f"{F_GUIDE}; {F_EXPL}")
row("Definitions",
    "Reported completed pathways are treatments",
    "no: the guidance lists six clock-stop routes, one of them first treatment; NHS England's explainer puts treatment starts at 50% of reported completed pathways in the WLMDS (2025)",
    "definitions; NHS England analysis", "does not hold", "contradicted by the definitions", "-", f"{F_GUIDE}; {F_EXPL}")
row("Definitions",
    "A pathway can close without anyone being treated",
    "yes: decision not to treat, patient declined, active monitoring, discharge after a missed appointment and death all stop the clock",
    "definitions", "holds", "stated in NHS England's clock-stop definitions", "-", f"{F_GUIDE}; {F_EXPL}")
row("Definitions",
    "Unreported removals are validation, or bookkeeping",
    "no: NHS England says they are not the same as validation removals, and are similar in nature to reported completions",
    "NHS England explainer", "does not hold", "contradicted by the explainer", "-", F_EXPL)
row("Definitions",
    "About half of exits were treatment starts, whether reported as completions or not (2025)",
    "explainer: 50% of reported completed pathways in the WLMDS, and 49% of the unreported removals that carried a reason code "
    "(a subset of 113,000 records), were treatment starts; WLMDS is management information",
    "NHS England analysis", "sourced, not rebuilt", "no published data to rebuild it from", "-", F_EXPL)
row("Definitions",
    "Why pathways close is an open question, and the regulator has asked NHS England to find out",
    "OSR (6 Nov 2026) encourages NHS England to continue work to understand and communicate why RTT pathways close; "
    "NHS England (21 Oct 2026) lists that analysis as planned, timetable dependent on resourcing",
    "correspondence", "holds", "read in both letters", "-", f"{F_OSR}; {F_NHSE}")

findings = pd.DataFrame(rows)
findings.insert(0, "id", range(1, len(findings) + 1))
findings.to_csv(OUT / "findings.csv", index=False)
findings[["id", "section", "claim_as_stated", "rebuilt", "verdict"]]

,id,section,claim_as_stated,rebuilt,verdict
0,1,Monthly flows,"Each month from April 2025 to September 2026, NHS England's figures show about 209,000...","209,442 (2025-08) to 351,871 (2026-03); median 260,926",holds
1,2,Monthly flows,Those unreported removals were about one exit in seven,median 14.4% of all exits; range 12.5% (2026-04) to 17.1% (2026-03); pooled over the w...,holds
2,3,Monthly flows,That is in line with NHS England's own long-run figure (14% of removals since 2021),window pooled 14.6% vs NHS England's stated 14%,holds
3,4,March 2026,March 2026 had the most unreported removals of any month in the window,"351,871, rank 1 of 18; 1.35x the window median (260,926)",holds
4,5,March 2026,March's spike is not a submission artefact: it shows among trusts that reported in bot...,"continuing trusts 345,405, rank 1 of 17; 1.35x their median (256,249)",holds
5,6,March 2026,"In March, unreported removals were also an unusually large share of exits",17.1% vs median 14.4%; rank 1 of 18 by share,holds
6,7,March 2026,"Arithmetic, not cause: had March's unreported removals been at the other months' media...","March excess over the other months' median: 91,572; the list fell 110,117; excess = 83...",holds
7,8,March 2026,March was a validation push,not observable: NHS England's explainer says unreported removals are not the same as v...,does not hold
8,9,September 2026,"In September 2026, unreported removals were above the window median","302,409 (1.16x the median), 15.3% of exits; rank 3 of 18",holds
9,10,September 2026,"The reported-providers figure for September 2026 (377,813) overstates unreported removals","reported 377,813 vs headline 302,409; Nottingham (RX1) stopped submitting, booking its...",holds


## Step 3 — Tally and a readable copy

In [5]:
order = ["holds", "true but misleading", "overstated", "does not hold", "sourced, not rebuilt"]
assert set(findings.verdict) <= set(order)
tally = findings.verdict.value_counts().reindex(order, fill_value=0)
md = ["# Flows piece — claims against the data", "",
      "Generated by `notebooks/11_gold_flows_findings.ipynb` from Gold 10 and the saved evidence. Do not edit by hand.", ""]
for sec, g in findings.groupby("section", sort=False):
    md += [f"## {sec}", "", g[["id", "claim_as_stated", "rebuilt", "basis", "verdict", "rule"]].to_markdown(index=False), ""]
md += ["## Tally", "", tally.rename("claims").to_frame().to_markdown(), ""]
(OUT / "findings.md").write_text("\n".join(md))
print(tally.to_string())
print("\ntop 10 continuing trusts by net unreported removals, May 2025 -", month_year(SEP))
print(top10.rename(lambda e: f"{e} {NAMES.get(e, '')}").round(0).to_string())
print("\ntop 10 March excesses over each trust's own median:")
print(ex_pos.head(10).rename(lambda e: f"{e} {NAMES.get(e, '')}").round(0).to_string())

verdict
holds                   14
true but misleading      0
overstated               0
does not hold            6
sourced, not rebuilt     1

top 10 continuing trusts by net unreported removals, May 2025 - September 2026
entity
RAL ROYAL FREE LONDON NHS FOUNDATION TRUST                    152724.0
RC9 BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST               150314.0
RAJ MID AND SOUTH ESSEX NHS FOUNDATION TRUST                  131196.0
RTE GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST            120513.0
RTD THE NEWCASTLE UPON TYNE HOSPITALS NHS FOUNDATION TRUST    107626.0
R0A MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                107050.0
RHW ROYAL BERKSHIRE NHS FOUNDATION TRUST                       98510.0
RM3 NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST                97128.0
RJ1 GUY'S AND ST THOMAS' NHS FOUNDATION TRUST                  86568.0
RR8 LEEDS TEACHING HOSPITALS NHS TRUST                         85725.0

top 10 March excesses over each trust's own median:
entity
